# Chapter 5 Lab: Dual Frames

This notebook accompanies **Chapter 5**. It builds the canonical dual,
the affine dual family, and Monte Carlo noise-stability machinery, then
works through all five lab exercises.

Run the setup cell first.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import null_space
from scipy.optimize import minimize_scalar

np.set_printoptions(precision=4, suppress=True)
%matplotlib inline

def frame_operator(F):
    return F @ F.T

def exact_frame_bounds(F):
    S = frame_operator(F)
    eigenvalues, eigenvectors = np.linalg.eigh(S)
    return float(eigenvalues[0]), float(eigenvalues[-1]), eigenvalues, eigenvectors

def canonical_dual(F):
    '''Synthesis matrix of the canonical dual frame: S^{-1} F.'''
    S = frame_operator(F)
    return np.linalg.solve(S, F)

def dual_family_direction(F):
    '''Orthonormal basis of ker(F), shape (m, m-n).'''
    return null_space(F)

def random_alternate_dual(F, scale=1.0, seed=0):
    n, m = F.shape
    N = dual_family_direction(F)
    rng = np.random.default_rng(seed)
    W = rng.standard_normal((n, N.shape[1])) * scale
    Z = W @ N.T
    G_tilde = canonical_dual(F)
    return G_tilde + Z

## Lab Exercise 5.1: The Canonical Dual Pipeline

Run `canonical\_dual(F)` on (a) triangular, (b) $\{h_1,h_2,h_3\}$, (c) square. Verify mutual duality: both `F\_tilde @ F.T` and `F @ F\_tilde.T` equal the identity.

In [ ]:
f1 = np.array([1.0, 0.0])
f2 = np.array([-0.5, np.sqrt(3)/2])
f3 = np.array([-0.5, -np.sqrt(3)/2])
F_tri = np.column_stack([f1, f2, f3])

h1, h2, h3 = np.array([1.0,0.0]), np.array([0.0,1.0]), np.array([1.0,1.0])
F_h = np.column_stack([h1, h2, h3])

angles4 = np.array([0, np.pi/2, np.pi, 3*np.pi/2])
F_sq = np.vstack([np.cos(angles4), np.sin(angles4)])

for name, F in [("Triangular", F_tri), ("{h1,h2,h3}", F_h), ("Square", F_sq)]:
    F_tilde = None  # TODO: canonical_dual(F)
    print(f"--- {name} ---")
    print(f"F_tilde =\n{F_tilde}")

    check1 = None  # TODO: F_tilde @ F.T
    check2 = None  # TODO: F @ F_tilde.T
    print(f"F_tilde @ F.T == I: {np.allclose(check1, np.eye(2))}")
    print(f"F @ F_tilde.T == I: {np.allclose(check2, np.eye(2))}\n")

F_tilde_tri = canonical_dual(F_tri)
print("Triangular frame check: F_tilde == (1/1.5)*F_tri?",
      np.allclose(F_tilde_tri, (1/1.5) * F_tri))

## Lab Exercise 5.2: Verifying the Dual Family

(a) Reproduce Example 5.2's alternate dual with $w=(0.1,-0.2)$. (b) Generate 10 random alternate duals; verify exact reconstruction. (c) Confirm $\|G\|_F^2$ always exceeds the canonical value.

In [ ]:
k = dual_family_direction(F_h)[:, 0]
print(f"ker(H) direction: {k}")

w_example = np.array([0.1, -0.2])
F_h_tilde = canonical_dual(F_h)
Z_example = None  # TODO: np.outer(w_example, k)
G_example = None  # TODO: F_h_tilde + Z_example

print(f"G @ F_h.T (should be I):\n{G_example @ F_h.T}")

rng = np.random.default_rng(1)
test_xs = rng.standard_normal((2, 5))

all_reconstruct_ok = True
for seed in range(10):
    G_rand = None  # TODO: random_alternate_dual(F_h, scale=1.0, seed=seed)
    for i in range(5):
        x = test_xs[:, i]
        c = F_h.T @ x
        x_hat = G_rand @ c
        if not np.allclose(x_hat, x, atol=1e-9):
            all_reconstruct_ok = False
print(f"\nAll 10 random duals reconstruct exactly for all 5 test vectors: {all_reconstruct_ok}")

canonical_energy = np.trace(F_h_tilde @ F_h_tilde.T)
print(f"\nCanonical ||F_tilde||_F^2 = {canonical_energy:.4f}")
for seed in range(10):
    G_rand = random_alternate_dual(F_h, scale=1.0, seed=seed)
    energy = np.trace(G_rand @ G_rand.T)
    print(f"seed={seed}: ||G||_F^2 = {energy:.4f}, exceeds canonical: {energy > canonical_energy}")

## Lab Exercise 5.3: Monte Carlo Stability Comparison

Vary the perturbation scale $0.05,0.2,0.5,1.0,2.0$; compute predicted MSE ($\sigma=0.3$) and confirm via $50{,}000$-trial Monte Carlo.

In [ ]:
sigma = 0.3
scales = [0.05, 0.2, 0.5, 1.0, 2.0]
x_fixed = np.array([1.0, -0.5])

predicted_mse = []
empirical_mse = []

for scale in scales:
    G = None  # TODO: random_alternate_dual(F_h, scale=scale, seed=0)
    pred = None  # TODO: sigma**2 * np.trace(G @ G.T)
    predicted_mse.append(pred)

    rng = np.random.default_rng(2)
    c_true = F_h.T @ x_fixed
    errs = []
    for _ in range(50_000):
        noise = rng.normal(0, sigma, size=3)
        x_hat = G @ (c_true + noise)
        errs.append(np.sum((x_hat - x_fixed)**2))
    empirical_mse.append(np.mean(errs))

plt.figure(figsize=(7,5))
plt.plot(scales, predicted_mse, 'o-', label='predicted MSE')
plt.plot(scales, empirical_mse, 's--', label='empirical MSE (Monte Carlo)')
plt.xlabel('perturbation scale')
plt.ylabel('mean squared error')
plt.legend()
plt.title('Predicted vs. empirical MSE as dual-frame perturbation grows')
plt.show()

for s, p, e in zip(scales, predicted_mse, empirical_mse):
    print(f"scale={s}: predicted={p:.4f}, empirical={e:.4f}, rel_diff={abs(p-e)/p:.3%}")

*Your practical guidance: what does this experiment suggest about choosing a dual frame when noise is unavoidable?* (replace this text)

## Lab Exercise 5.4: Dual Frames of a Basis Are Unique

For basis $\{(2,1),(1,3)\}$, compute the canonical dual, verify $\ker F$ is trivial, and conclude the canonical dual is the only dual.

In [ ]:
v1 = np.array([2.0, 1.0])
v2 = np.array([1.0, 3.0])
F_basis = np.column_stack([v1, v2])

F_basis_tilde = None  # TODO: canonical_dual(F_basis)
print(f"Canonical dual:\n{F_basis_tilde}")

kernel = None  # TODO: dual_family_direction(F_basis)
print(f"ker(F_basis) shape: {kernel.shape}  (expect (2, 0), i.e. trivial kernel)")

biorthog = None  # TODO: F_basis.T @ F_basis_tilde
print(f"Biorthogonality check (should be I):\n{biorthog}")

*Your one-sentence explanation connecting this to the familiar biorthogonal-basis fact:* (replace this text)

## Lab Exercise 5.5 (Optional, Challenge): Sparsity-Promoting Duals

(a) Parametrize $c'(t)=c+tn$, plot $\|c'(t)\|_1$. (b) Find the L1-minimizing $t$; is the result sparse?

In [ ]:
x = np.array([1.0, -0.5])
c = F_h.T @ x
n_vec = dual_family_direction(F_h)[:, 0]

print(f"c = {c}")
print(f"n (spans ker H) = {n_vec}")

def c_prime(t):
    return None  # TODO: c + t * n_vec

ts = np.linspace(-2, 2, 400)
l1_norms = np.array([None for t in ts])  # TODO: np.sum(np.abs(c_prime(t))) for each t

plt.figure(figsize=(7,4))
plt.plot(ts, l1_norms)
plt.xlabel('t')
plt.ylabel(r"$\|c'(t)\|_1$")
plt.title('L1 norm of the coefficient family as a function of t')
plt.show()

def l1_objective(t):
    return np.sum(np.abs(c_prime(t)))

result = None  # TODO: minimize_scalar(l1_objective, bounds=(-2,2), method='bounded')
t_star = result.x
c_star = c_prime(t_star)
print(f"\nt* = {t_star:.6f}")
print(f"c'(t*) = {c_star}")
print(f"Is c'(t*) sparse (has a near-zero entry)? {np.any(np.abs(c_star) < 1e-6)}")

*Your observation: does the L1-minimizing coefficient vector have an exact zero entry?* (replace this text)